In [10]:
#NORMALIZARION = NONW

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor()  
])


trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

print("MNIST Loaded successfully!")
print(f"Number of training batches (batch size 64): {len(trainloader)}")

MNIST Loaded successfully!
Number of training batches (batch size 64): 938


In [11]:
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = MNISTNet()
print(model)

MNISTNet(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5  

for epoch in range(epochs):
    running_loss = 0.0

    for i, (images, labels) in enumerate(trainloader):
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(trainloader):.4f}")

print("Training complete!")

Epoch [1/5], Loss: 0.3441
Epoch [2/5], Loss: 0.1579
Epoch [3/5], Loss: 0.1092
Epoch [4/5], Loss: 0.0832
Epoch [5/5], Loss: 0.0647
Training complete!


In [13]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy of the network on the 10,000 test images: {accuracy:.2f}%")

Accuracy of the network on the 10,000 test images: 97.47%


In [14]:
MODEL_PATH = "/mnist_Number_Image_model.pth"
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model successfully saved to {MODEL_PATH}!")

Model successfully saved to ../Models/mnist_Number_Image_model.pth!


In [15]:
import torchvision
import torchvision.transforms as transforms

train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

to_pil = transforms.ToPILImage()

for i in range(5):
    img_tensor, label = train_dataset[i]
    pil_img = to_pil(img_tensor)
    filename = f"mnist_real_sample_{i}_label_{label}.png"
    pil_img.save(filename)
    print(f"Saved {filename} (True Label: {label})")

print("Done")

Saved mnist_real_sample_0_label_5.png (True Label: 5)
Saved mnist_real_sample_1_label_0.png (True Label: 0)
Saved mnist_real_sample_2_label_4.png (True Label: 4)
Saved mnist_real_sample_3_label_1.png (True Label: 1)
Saved mnist_real_sample_4_label_9.png (True Label: 9)
Done! Check your folder for the 'mnist_real_sample_*.png' files.
